In [1]:
import glob, json, os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

hits = glob.glob("/kaggle/input/**/predictions.csv", recursive=True)
assert hits, "predictions.csv not found. Add the output of xray-02-train as an input."
preds = pd.read_csv(hits[0])
val = preds[preds.split == "val"].reset_index(drop=True)
test = preds[preds.split == "test"].reset_index(drop=True)
print("Validation patients:", len(val), "| Test patients:", len(test))

y = test.target.values                          
p = test.prob.values                        
ap = (test["view"] == "AP").astype(float).values   
cls = test["class"].values
groups = {
    "Bedside X-rays only (AP)": ap == 1,
    "Standard X-rays only (PA)": ap == 0,
    "Pneumonia vs Normal": cls != "No Lung Opacity / Not Normal",
    "Pneumonia vs Not Normal (other problems)": cls != "Normal",
}

def scores(i):
    """All the scores, computed on the patients whose row numbers are in i."""
    out = {"Model: ROC AUC, all test patients": roc_auc_score(y[i], p[i]),
           "Model: PR AUC, all test patients": average_precision_score(y[i], p[i]),
           "Shortcut (X-ray view only): ROC AUC": roc_auc_score(y[i], ap[i])}
    out["Model minus shortcut"] = out["Model: ROC AUC, all test patients"] - out["Shortcut (X-ray view only): ROC AUC"]
    for name, mask in groups.items():
        j = i[mask[i]]
        out["Model: ROC AUC, " + name] = roc_auc_score(y[j], p[j])
    return out

everyone = np.arange(len(test))
point = scores(everyone)

rng = np.random.default_rng(42)
B = 2000
boot = pd.DataFrame([scores(rng.integers(0, len(test), len(test))) for _ in range(B)])

table = pd.DataFrame({"estimate": point, "ci_low": boot.quantile(0.025), "ci_high": boot.quantile(0.975)}).round(3)
table["patients"] = [len(test)] * 4 + [int(m.sum()) for m in groups.values()]
table["pneumonia"] = [int(y.sum())] * 4 + [int(y[m].sum()) for m in groups.values()]
print("\nTest set results with 95% confidence intervals (2,000 bootstrap resamples):")
print(table.to_string())
table.to_csv("/kaggle/working/classifier_metrics.csv")

Validation patients: 2669 | Test patients: 5337

Test set results with 95% confidence intervals (2,000 bootstrap resamples):
                                                          estimate  ci_low  ci_high  patients  pneumonia
Model: ROC AUC, all test patients                            0.884   0.874    0.894      5337       1203
Model: PR AUC, all test patients                             0.714   0.689    0.742      5337       1203
Shortcut (X-ray view only): ROC AUC                          0.703   0.689    0.717      5337       1203
Model minus shortcut                                         0.181   0.167    0.195      5337       1203
Model: ROC AUC, Bedside X-rays only (AP)                     0.835   0.819    0.851      2401        920
Model: ROC AUC, Standard X-rays only (PA)                    0.867   0.844    0.888      2936        283
Model: ROC AUC, Pneumonia vs Normal                          0.975   0.969    0.980      2973       1203
Model: ROC AUC, Pneumonia vs Not No

In [2]:
from sklearn.metrics import brier_score_loss

fpr, tpr, thr = roc_curve(val.target, val.prob)
T = float(thr[np.argmax(tpr - fpr)])          # the point with the best balance of sensitivity and specificity
flag = p >= T
tp, fp = int((flag & (y == 1)).sum()), int((flag & (y == 0)).sum())
fn, tn = int((~flag & (y == 1)).sum()), int((~flag & (y == 0)).sum())
print(f"Threshold chosen on validation set: {T:.3f}")
print(f"Test confusion matrix: TP {tp} | FP {fp} | FN {fn} | TN {tn}")
print(f"Sensitivity {tp / (tp + fn):.3f} | Specificity {tn / (tn + fp):.3f} | Precision {tp / (tp + fp):.3f}")

print("\nShare of each class flagged as pneumonia:")
print(pd.Series(flag).groupby(cls).mean().round(3))

prev = y.mean()
print(f"\nBrier score {brier_score_loss(y, p):.4f} (always guessing the average rate would score {prev * (1 - prev):.4f}; lower is better)")
bins = pd.qcut(p, 10, duplicates="drop")
calib = pd.DataFrame({"predicted": p, "actual": y}).groupby(bins, observed=True).agg(
    patients=("actual", "size"), mean_predicted=("predicted", "mean"), actual_rate=("actual", "mean")).round(3)
print("\nCalibration (test patients in 10 equal groups, lowest to highest predicted probability):")
print(calib.to_string(index=False))
calib.to_csv("/kaggle/working/calibration_table.csv", index=False)

# 3. Does it work equally well for different patients?
age_band = pd.cut(test.age.where(test.age <= 100), [0, 39, 59, 100], labels=["under 40", "40 to 59", "60 and over"])
print("\nROC AUC by patient group:")
for name, col in [("sex", test.sex), ("age", age_band)]:
    for g in sorted(col.dropna().unique(), key=lambda v: ["under 40", "40 to 59", "60 and over", "F", "M"].index(v)):
        m = (col == g).values
        print(f"  {name} {g}: {roc_auc_score(y[m], p[m]):.3f}  ({int(m.sum())} patients, {int(y[m].sum())} with pneumonia)")

json.dump({"threshold": T}, open("/kaggle/working/threshold.json", "w"))
print("\nSaved:", sorted(f for f in os.listdir("/kaggle/working") if not f.startswith(".")))

Threshold chosen on validation set: 0.212
Test confusion matrix: TP 959 | FP 838 | FN 244 | TN 3296
Sensitivity 0.797 | Specificity 0.797 | Precision 0.534

Share of each class flagged as pneumonia:
Lung Opacity                    0.797
No Lung Opacity / Not Normal    0.341
Normal                          0.018
dtype: float64

Brier score 0.1112 (always guessing the average rate would score 0.1746; lower is better)

Calibration (test patients in 10 equal groups, lowest to highest predicted probability):
 patients  mean_predicted  actual_rate
      534           0.001        0.009
      534           0.003        0.015
      535           0.007        0.017
      532           0.014        0.041
      534           0.032        0.084
      533           0.081        0.148
      534           0.194        0.253
      533           0.395        0.328
      534           0.672        0.528
      534           0.923        0.830

ROC AUC by patient group:
  sex F: 0.884  (2285 patients, 488